# A first 1D photonic-crystal calculation: Plane Wave Expansion

**Goal:** turn a repeating pair of dielectric layers into a photonic band diagram, one understandable step at a time.

Upload this `.ipynb` file at [Google Colab](https://colab.research.google.com/), then run the cells from top to bottom. A standard Python runtime is enough; no GPU, external files, or account credentials are needed. NumPy and Matplotlib are the only libraries used. To keep your own editable copy, save the notebook to your Google Drive.

We will build the unit cell, calculate Fourier coefficients of inverse relative permittivity, assemble a magnetic-field eigenproblem, and find frequency intervals with no propagating Bloch mode. The FFT comparison is a separate validation of the analytic coefficients.

**Model limits:** an infinite stack of planar layers, normal incidence (no wavevector parallel to the layers), real positive scalar permittivity, relative permeability $\mu_r=1$, and no absorption or material dispersion. The two transverse polarizations have the same spectrum here. This is not a finite-stack transmission calculation, an omnidirectional gap calculation, or a COMSOL model.


## 1. Imports and numerical settings
`numpy` handles arrays and eigenvalues; `matplotlib` draws the figures. Python uses `1j` for $i=\sqrt{-1}$. Complex Fourier coefficients are expected even though the materials are real.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize': (9, 4.5), 'font.size': 11,
                     'axes.grid': True, 'grid.alpha': 0.25})
c0 = 299_792_458.0  # vacuum speed of light, m/s
print('NumPy version:', np.__version__)


## 2. Define a binary unit cell
Let $a$ be the repeat length and $f$ the fraction occupied by material 1:
$$\epsilon_r(x)=\begin{cases}\epsilon_1,&0\le x<fa,\\\epsilon_2,&fa\le x<a.\end{cases}$$
Repeat this cell for all $x$. For our nonmagnetic, lossless model, $\epsilon_j=n_j^2$.
Hereafter **$\epsilon$ means relative permittivity**, which is dimensionless; absolute permittivity is $\epsilon_0\epsilon_r$ and is not inserted into this matrix.

Start with **illustrative glass-like material, $n_1=1.50$, and air-like material, $n_2=1.00$**. These rounded, nondispersive teaching values are not a measured material dataset or inherited project parameters. Choose $f=0.5$ and $a=500$ nm. Changing $a$ rescales physical frequencies but not the normalized band diagram in this nondispersive model.


In [ ]:
# Edit these four physical inputs, then rerun all cells below.
n1, n2 = 1.50, 1.00
f = 0.50                  # filling fraction, not frequency
a = 500e-9                # period in metres
eps1, eps2 = n1**2, n2**2
eta1, eta2 = 1/eps1, 1/eps2
assert a > 0 and n1 > 0 and n2 > 0 and 0 < f < 1
print(f'epsilon_1={eps1:.4f}, epsilon_2={eps2:.4f}')
print(f'Layer thicknesses: {f*a*1e9:.1f} nm and {(1-f)*a*1e9:.1f} nm')

u = np.linspace(0, 2, 2001, endpoint=False)  # u=x/a; two periods
eps_x = np.where(np.mod(u, 1) < f, eps1, eps2)
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 5))
axes[0].plot(u, eps_x); axes[0].set_ylabel(r'$\epsilon_r(x)$')
axes[1].plot(u, 1/eps_x); axes[1].set_ylabel(r'$\eta(x)=1/\epsilon_r(x)$')
axes[1].set_xlabel(r'Position $x/a$')
fig.suptitle('A binary unit cell repeated twice'); fig.tight_layout(); plt.show()


## 3. Reciprocal lattice vectors and analytic Fourier coefficients
The reciprocal vectors are $G_m=2\pi m/a$, where $m$ is an integer and $G_m$ has units rad/m. Expand
$$\eta(x)=\frac{1}{\epsilon_r(x)}=\sum_{p=-\infty}^{\infty}\eta_p e^{iG_p x},\qquad
\eta_p=\frac1a\int_0^a\eta(x)e^{-iG_p x}\,dx.$$
The integer subscript $p$ is shorthand for the reciprocal vector $G_p$. We use $p$ for a coefficient index and $m,m'$ for field-basis indices.

For $p=0$, the exponential is one:
$$\boxed{\eta_0=f\eta_1+(1-f)\eta_2=\frac f{\epsilon_1}+\frac{1-f}{\epsilon_2}.}$$
In this equation only, $\eta_1,\eta_2$ on the right denote **material values**; in code these are `eta1`, `eta2`, whereas coefficients are returned by `eta_coeff(p)`. The mean of $1/\epsilon$ is generally **not** the reciprocal of the mean of $\epsilon$.

For $p\ne0$, the constant background integrates to zero over a period, leaving
$$\eta_p=\frac{\eta_1-\eta_2}{a}\int_0^{fa}e^{-iG_p x}dx
=\boxed{(\eta_1-\eta_2)\frac{1-e^{-i2\pi pf}}{i2\pi p}}.$$
Using $1-e^{-i2\theta}=2i e^{-i\theta}\sin\theta$, a numerically convenient expression is
$$\eta_p=\eta_2\delta_{p0}+(\eta_1-\eta_2)f\,\operatorname{sinc}(pf)e^{-i\pi pf},$$
where NumPy defines $\operatorname{sinc}(z)=\sin(\pi z)/(\pi z)$ and $\operatorname{sinc}(0)=1$. This expression handles zero without dividing by zero. The phase comes from placing layer 1 at the start of the cell. Do not discard its imaginary part. All coefficients are dimensionless.


In [ ]:
def eta_coeff(p, eps_a=eps1, eps_b=eps2, fill=f):
    """Analytic coefficient eta_p; accepts a scalar or array of integer indices."""
    p = np.asarray(p)
    contrast = 1/eps_a - 1/eps_b
    return ((p == 0)/eps_b + contrast*fill*np.sinc(p*fill)
            * np.exp(-1j*np.pi*p*fill))

p_demo = np.arange(-4, 5)
for p, value in zip(p_demo, eta_coeff(p_demo)):
    print(f'p={p:2d}: {value.real:+.7f} {value.imag:+.7f}j')
assert np.allclose(eta_coeff(0), f/eps1+(1-f)/eps2)
assert np.allclose(eta_coeff(-p_demo), np.conj(eta_coeff(p_demo)))
pnz = p_demo[p_demo != 0]
direct = (eta1-eta2)*(1-np.exp(-2j*np.pi*pnz*f))/(2j*np.pi*pnz)
assert np.allclose(eta_coeff(pnz), direct)
print('Mean, conjugate symmetry, and nonzero-index formula: PASS')


## 4. Validate the coefficients with an FFT
On $N$ equally spaced samples $x_j=ja/N$, the integral becomes
$$\eta_p\approx\frac1N\sum_{j=0}^{N-1}\eta(x_j)e^{-2\pi i pj/N}.$$
This is `np.fft.fft(samples)/N`. We sample **$1/\epsilon$**, not $\epsilon$ followed by inversion of its Fourier coefficients. There is no duplicated endpoint, window, or `fftshift`. Negative indices live at the end of the FFT array; `p % N` retrieves them.

Discontinuities give finite sampling error. An interface not exactly on the grid can also change the sampled filling fraction. Compare absolute error because some exact coefficients vanish. FFT agreement tests the material coefficients; it does not establish convergence of the band calculation.


In [ ]:
p_check = np.arange(-30, 31)
exact = eta_coeff(p_check)
fft_errors = []
sample_sizes = [256, 1024, 4096, 16384]
for N in sample_sizes:
    assert 2*np.max(np.abs(p_check)) < N  # avoid Nyquist ambiguity
    x_over_a = np.arange(N)/N
    samples = np.where(x_over_a < f, eta1, eta2)
    fft_coeffs = np.fft.fft(samples)/N
    estimated = fft_coeffs[p_check % N]
    err = np.max(np.abs(estimated-exact))
    fft_errors.append(err)
    print(f'N={N:5d}: max absolute coefficient error = {err:.3e}')
# A conservative finite-grid check, scaled to the material contrast.
assert fft_errors[-1] < 10*abs(eta1-eta2)/sample_sizes[-1] + 1e-12
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(p_check, exact.real, 'o', label='analytic real')
axes[0].plot(p_check, estimated.real, '+', label='FFT real')
axes[0].plot(p_check, exact.imag, '.', label='analytic imaginary')
axes[0].plot(p_check, estimated.imag, 'x', label='FFT imaginary')
axes[0].set(xlabel='Coefficient index p', ylabel='Coefficient'); axes[0].legend()
axes[1].loglog(sample_sizes, np.maximum(fft_errors, 1e-16), 'o-')
axes[1].set(xlabel='FFT samples N', ylabel='Maximum absolute error')
fig.tight_layout(); plt.show()


## 5. Bloch wavevector and the first Brillouin zone
Bloch's theorem writes a transverse magnetic field as
$$H(x)=e^{ikx}u_k(x)=\sum_m h_m e^{i(k+G_m)x},\qquad u_k(x+a)=u_k(x).$$
$k$ is the Bloch wavevector: translating by one cell multiplies $H$ by $e^{ika}$. It is not the frequency or an index. Since adding a reciprocal vector gives an equivalent Bloch label, use the first Brillouin zone $-\pi/a\le k\le\pi/a$.

Define $q=ka/(2\pi)$, so $q\in[-1/2,1/2]$. We retain $m=-M,\ldots,M$, giving $2M+1$ plane waves. `M` controls field resolution; the number of sampled $k$ values controls how finely we trace each band. These are different convergence parameters.


In [ ]:
M = 20                      # 41 plane waves
num_bands = 5                # lowest five eigenfrequencies
q_values = np.linspace(-0.5, 0.5, 201)  # includes q=0 and zone edges
m = np.arange(-M, M+1)
G = 2*np.pi*m/a              # physical reciprocal vectors, rad/m
k_values = 2*np.pi*q_values/a
assert num_bands < len(m)
print(f'{len(m)} plane waves; {len(q_values)} Bloch wavevectors')
print(f'First Brillouin zone: [{-np.pi/a:.3e}, {np.pi/a:.3e}] rad/m')


## 6. From the H-field equation to a matrix
For this geometry, Maxwell's equations reduce to
$$-\frac{d}{dx}\left[\eta(x)\frac{dH}{dx}\right]=\left(\frac\omega c\right)^2H.$$
Insert the two Fourier series. The inner derivative contributes $i(k+G_{m'})$; multiplication by $\eta$ couples harmonics with $p=m-m'$; the outer derivative and minus sign contribute $-i(k+G_m)$. Thus
$$\sum_{m'} A_{mm'}h_{m'}=(\omega/c)^2h_m,\quad
\boxed{A_{mm'}=(k+G_m)(k+G_{m'})\eta_{m-m'}.}$$
The physical matrix $A$ and its eigenvalues have units $\mathrm{m}^{-2}$ (radians are dimensionless). **Do not confuse $\eta_{m-m'}$ with $\eta_m-\eta_{m'}$.** Differences range from $-2M$ to $2M$, even though field indices range only from $-M$ to $M$.

For easier arithmetic, solve the dimensionless matrix
$$B=\left(\frac a{2\pi}\right)^2 A,\qquad B_{mm'}=(q+m)(q+m')\eta_{m-m'}.$$
Its eigenvalues are $\lambda=\Omega^2$, where
$$\boxed{\Omega=\frac{\omega a}{2\pi c}=\frac{\nu a}{c}=\frac a{\lambda_0}.}$$
$\omega$ is angular frequency (rad/s), $\nu=\omega/(2\pi)$ is ordinary frequency (Hz), and $\lambda_0$ is vacuum wavelength. Therefore `sqrt(eigenvalue_of_B)` is already the normalized frequency: **do not divide it by another $2\pi$**.

Real positive permittivity makes $B$ Hermitian and positive semidefinite. Use a Hermitian eigenvalue solver. Tiny negative roundoff can be clipped; a substantially negative eigenvalue must raise an error.


In [ ]:
def pwe_matrix(q, cutoff, eps_a=eps1, eps_b=eps2, fill=f):
    indices = np.arange(-cutoff, cutoff+1)
    differences = indices[:, None] - indices[None, :]
    eta_matrix = eta_coeff(differences, eps_a, eps_b, fill)
    momenta = q + indices
    return momenta[:, None]*eta_matrix*momenta[None, :]

def solve_bands(q_grid, cutoff, count, eps_a=eps1, eps_b=eps2, fill=f):
    assert 1 <= count <= 2*cutoff+1
    result = np.empty((len(q_grid), count))
    for j, q in enumerate(q_grid):
        B = pwe_matrix(q, cutoff, eps_a, eps_b, fill)
        assert np.allclose(B, B.conj().T, rtol=1e-12, atol=1e-12)
        eigenvalues = np.linalg.eigvalsh(B)  # ascending real eigenvalues
        tolerance = 1e-11*max(1.0, np.max(np.abs(eigenvalues)))
        if eigenvalues[0] < -tolerance:
            raise ValueError('Unexpected negative eigenvalue: check matrix construction')
        result[j] = np.sqrt(np.maximum(eigenvalues[:count], 0.0))
    return result

bands = solve_bands(q_values, M, num_bands)
print('Array shape (k points, bands):', bands.shape)
# Check the physical matrix and the dimensionless matrix at the same q.
q_test = 0.23
k_test = q_test*2*np.pi/a
A = ((k_test+G)[:, None]*(k_test+G)[None, :]
     *eta_coeff(m[:, None]-m[None, :]))
assert np.allclose(A*(a/(2*np.pi))**2, pwe_matrix(q_test, M))
print('Hermiticity, positivity, and unit conversion: PASS')


## 7. Sanity checks before trusting the plot
At $k=0$ a spatially constant $H$ has zero derivative, so the lowest frequency is zero. Reciprocity gives the same frequencies at $k$ and $-k$. In a uniform medium, the exact folded dispersion is $\Omega_m=|q+m|/n$, with no true gaps. These checks catch errors that a plausible-looking plot can hide.


In [ ]:
zero_index = np.argmin(np.abs(q_values))
assert bands[zero_index, 0] < 1e-6
assert np.allclose(bands, bands[::-1], atol=1e-6)
q_uniform = np.array([-0.5, -0.17, 0.0, 0.31, 0.5])
n_uniform = 1.5
uniform = solve_bands(q_uniform, M, num_bands, n_uniform**2, n_uniform**2)
expected = np.sort(np.abs(q_uniform[:, None]+m[None, :])/n_uniform, axis=1)[:, :num_bands]
assert np.allclose(uniform, expected, atol=1e-7)
print('Zero-frequency mode, k symmetry, and uniform-medium dispersion: PASS')


## 8. Find gaps across the entire sampled zone
For adjacent sorted bands $b$ and $b+1$, calculate
$$\Omega_L=\max_k\Omega_b(k),\qquad\Omega_U=\min_k\Omega_{b+1}(k).$$
A positive width $\Delta\Omega=\Omega_U-\Omega_L$ means a candidate gap **over this sampled grid**. A separation at just one $k$ is insufficient. We report the relative gap width $\Delta\Omega/[(\Omega_L+\Omega_U)/2]$ as a percentage. A small detection threshold avoids reporting roundoff as a gap; it is not a convergence guarantee.

These are gaps for propagation normal to the layers. Only gaps between the computed bands are tested, and their edges need convergence checks below.


In [ ]:
def find_gaps(values, threshold=1e-5):
    gaps = []
    for b in range(values.shape[1]-1):
        lower = values[:, b].max()
        upper = values[:, b+1].min()
        if upper-lower > threshold:
            gaps.append((b+1, lower, upper))  # human band numbers start at 1
    return gaps

gaps = find_gaps(bands)
print('Band pair    lower       upper       width       relative width')
for b, lower, upper in gaps:
    print(f'{b:2d}-{b+1:<2d}      {lower:.6f}    {upper:.6f}    '
          f'{upper-lower:.6f}    {100*(upper-lower)/((upper+lower)/2):.2f}%')
if not gaps:
    print('No gaps detected between these bands on this grid.')

fig, ax = plt.subplots(figsize=(9, 6))
for b in range(num_bands):
    ax.plot(q_values, bands[:, b], label=f'Band {b+1}')
for j, (b, lower, upper) in enumerate(gaps):
    ax.axhspan(lower, upper, color='gold', alpha=0.3,
               label='Candidate gap' if j == 0 else None)
ax.set(xlabel=r'Bloch wavevector $q=ka/(2\pi)$',
       ylabel=r'Normalized frequency $\Omega=\omega a/(2\pi c)$',
       title=f'1D H-field PWE: n1={n1:g}, n2={n2:g}, f={f:g}, M={M}',
       xlim=(-0.5, 0.5), ylim=(0, bands.max()*1.04))
ax.set_xticks([-0.5, -0.25, 0, 0.25, 0.5]); ax.legend(loc='upper right', fontsize=9)
fig.tight_layout(); plt.show()


## 9. Convergence: increase the basis and refine the k grid
A sharp dielectric interface needs many harmonics. Analytic coefficients remove quadrature error but not field-basis truncation error. Compare the first gap for $M=10,20,40,80$, then refine the grid for the largest basis. The printed changes are empirical error indicators, not rigorous bounds. Higher bands usually require more plane waves.

This section focuses on the first gap. Before reporting any other gap in research, repeat the convergence study for that band pair and choose tolerances suitable for your work. Increase both limits further if the changes are too large.


In [ ]:
q_convergence = np.linspace(-0.5, 0.5, 101)
previous = None
convergence_edges = []
for cutoff in [10, 20, 40, 80]:
    trial = solve_bands(q_convergence, cutoff, 2)
    edges = np.array([trial[:, 0].max(), trial[:, 1].min()])
    change = np.nan if previous is None else np.max(np.abs(edges-previous))
    print(f'M={cutoff:2d}, waves={2*cutoff+1:3d}: '
          f'edges={edges[0]:.7f}, {edges[1]:.7f}; max change={change:.3e}')
    convergence_edges.append(edges)
    previous = edges

q_fine = np.linspace(-0.5, 0.5, 401)
fine = solve_bands(q_fine, 80, 2)
fine_edges = np.array([fine[:, 0].max(), fine[:, 1].min()])
print('101 -> 401 k points, M=80: edge change =', np.max(np.abs(fine_edges-previous)))
print('M=20 plot vs M=80 refined first-gap edges:',
      np.array([bands[:, 0].max(), bands[:, 1].min()])-fine_edges)


## 10. Convert a gap to ordinary frequency and vacuum wavelength
Use $\nu=\Omega c/a$ and $\lambda_0=a/\Omega$. Since wavelength decreases as frequency increases, the lower wavelength edge comes from the **upper** frequency edge. This example uses the refined first-gap estimate from the preceding cell, not the coarser plotted values. Scaling to a real device would also require checking material dispersion and loss.


In [ ]:
lower, upper = fine_edges
if upper > lower + 1e-5 and lower > 0:
    print(f'Refined first gap: Omega = {lower:.7f} to {upper:.7f}')
    print(f'Frequency interval: {lower*c0/a/1e12:.2f} to {upper*c0/a/1e12:.2f} THz')
    print(f'Vacuum wavelength interval: {a/upper*1e9:.2f} to {a/lower*1e9:.2f} nm')
else:
    print('No positive first gap to convert.')


## 11. How the code maps to the physics

| Code block | Physics | What to inspect |
|---|---|---|
| Imports | Numerical and plotting tools | No simulation package is hidden here |
| `n1`, `n2`, `f`, `a` | Materials and layer thicknesses | $\epsilon_r=n^2$, lengths in metres |
| `eps_x`, `1/eps_x` | Real-space dielectric profile | Repeated binary layers |
| `eta_coeff(p)` | Analytic integral of inverse permittivity | Mean at zero, complex phase, conjugate symmetry |
| `fft(samples)/N` | Discrete approximation to that integral | Absolute errors shrink with resolution |
| `m`, `G`, `q_values` | Fourier basis and Bloch labels | Basis size and k-grid are independent |
| `differences` | Harmonic coupling by the periodic material | It is $m-m'$, not a difference of coefficient values |
| `pwe_matrix` | Dimensionless H-field differential operator | Hermitian, nonnegative spectrum |
| `eigvalsh` then square root | Eigenvalues $\Omega^2$ become frequencies | Each column is a sorted band, not a tracked eigenvector |
| Sanity checks | Known physical limits | Uniform dispersion, zero mode, reciprocity |
| `find_gaps` | Intervals missing from all sampled k | Max of lower band versus min of upper band |
| Convergence loop | Accuracy of the truncated expansion | Changes in band edges as M and k sampling increase |
| Frequency conversion | Relating scale-free bands to a chosen period | Hz versus rad/s; reversed wavelength bounds |

**Suggested experiments:** set `n1=n2` to remove the dielectric contrast; change `f`; double `a`; or increase `M`. After changing parameters, rerun all cells from the top because function defaults capture values when their definition cell runs. With equal indices there should be no physical gaps, and FFT error should be at roundoff. Doubling `a` leaves $\Omega$ unchanged and halves frequency in Hz. A different coordinate origin changes coefficient phases but should leave converged bands unchanged.

**What we have not claimed:** finite-M results are not exact, FFT agreement alone does not validate the full solver, and these one-direction gaps do not imply a gap for every angle in a three-dimensional structure.

### Further reading and API references
- [MIT Photonic Bands documentation: Introduction](https://mpb.readthedocs.io/en/latest/Introduction/) — background on computing electromagnetic eigenmodes in periodic dielectric structures. This notebook implements its own small 1D solver and does not call MPB.
- [NumPy discrete Fourier transform conventions](https://numpy.org/doc/stable/reference/routines.fft.html) — forward-transform sign, scaling, and index ordering used in section 4.
- [NumPy `eigvalsh`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.eigvalsh.html) — Hermitian eigenvalues in ascending order used in section 6.
